Q.1 What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

Limitations of MapReduce:
-Heavy disk I/O because intermediate results are written to disk after each stage.
-Slow performance for iterative algorithms (e.g., machine learning).
-Complex coding model requiring multiple Map and Reduce jobs.

SPARK is preferred becauses it uses in-memory processing. Faster execution than MapReduce.

Q.2 Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

Spark stores intermediate data in RAM rather than repeatedly writing it to disk.

For machine learning algorithms (e.g., Logistic Regression, K-Means), the same dataset is processed multiple times.

Q.3 Remove all duplicate rows based on user_id and transaction_date.


In [ ]:
df_clean = df.dropDuplicates(["user_id", "transaction_date"])

Q.4  Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

In [ ]:
from pyspark.sql.functions import avg

result = (
    df_sales
    .filter(df_sales.region == "West")
    .groupBy("product_category")
    .agg(avg("sale_amount").alias("avg_sale_amount"))
)

result.show()

Q.5 What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

.na.drop() - Removes rows containing null values.
.na.fill() - Replaces null values with specified values.

In [ ]:
df_filled = df.na.fill({"status": "Unknown"})

Q.6 Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

In [ ]:
from pyspark.sql.functions import count

result = (
    df.groupBy("city")
      .agg(count("*").alias("city_count"))
      .filter("city_count > 100")
)

result.show()

Q.7 How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

Spark DataFrames are immutable, meaning they can not be modified in place. Every transformation creates a new DataFrame.

Ex. Dropping a column.

In [ ]:
df_new = df.drop("temp_column")

Renaming a column

In [ ]:
df_new = df.withColumnRenamed("old_name", "new_name")

Q.8 Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

In [ ]:
result = df.filter(
    (df.age.between(18, 30)) &
    (df.subscription == "Premium")
)

result.show()

Q.9 When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

Handling null values before performing mathematical aggregations because it helps in preventing incorrect calculations, avoid unexpected missing results, improve data quality and consistency.

In [ ]:
df = df.na.fill({"sales": 0})

In [ ]:
# Before Calculating
df.selectExpr("avg(sales)").show()

Q.10 Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

df = (
    df.withColumn(
        "event_time",
        col("raw_timestamp").cast(TimestampType())
    )
    .drop("raw_timestamp")
)

df.show()

Q.11 Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

Shuffle is the process of redistributing data across partitions so that related records end up on the same executor.

In [ ]:
# example .
df.groupBy("city").count();

It is wide transformation because data moves between partitions and across executors.

Q.12 Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [ ]:
from pyspark.sql.functions import col

df_clean = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

df_clean.show()

Q.13 How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column? can you give me ans of this questions

In [ ]:
from pyspark.sql.functions import min, max, mean

result = df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    mean("price").alias("avg_price")
)

result.show()

Q.14 In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

Using inferSchema=true can lead to incorrect data type detection when date formats are inconsistent.
Risks:
-Wrong Data Type Assignment:
Spark may infer a date column as StringType instead of DateType if some values don't match the expected format.
-Null Values After Conversion:
Invalid or mixed date formats may be converted to null during parsing.

Q.15 Write a final processing pipeline that: Filters out duplicates, Fills null prices with 0, Groups by store_id to calculate total revenue.

In [ ]:
from pyspark.sql.functions import sum

result = (
    df
    .dropDuplicates()                # Remove duplicate rows
    .na.fill({"price": 0})           # Replace null prices with 0
    .groupBy("store_id")             # Group by store
    .agg(sum("price").alias("total_revenue"))
)

result.show()